# 4b · MIL training — instance-level, the instrument for Q2

> ⛔ **This stage is a stub and must stay one until the blanks in §2 are filled.** The architecture is
> not built, and the criterion that decides whether it succeeded is deliberately written *before* any
> run exists. A structured-looking result read after the fact is not evidence
> ([TODO](../docs/TODO.md), *Agreed plan, Step 2*).

## What this notebook is for

**Q2 — does within-line transcriptional heterogeneity survive into a model's per-cell response
predictions?** Scoped this way by Selin, 13.08.2026.

Concretely: do cells of one cell line differ enough in their representation — `pca` or `scgpt` — for a
model to assign them different response values, reproducibly and not as an artifact? Kinker et al.
(2020) documented recurrent heterogeneity programs in this dataset, but whether that heterogeneity
survives *our* preprocessing and embedding is **not assumed** — stage 0 measures it, and stage 0 can
fail.

It is the question the project is built around, because relapse is driven by rare surviving
subpopulations rather than by the average cell.

It is **structurally unanswerable** under `4a_percell_training`'s per-cell model: every cell of a line
carries that line's label, so the objective penalises exactly the within-line variation Q2 asks about
([Step 03](../docs/steps/03-model-and-training-design.md#every-cell-of-a-line-carries-the-identical-label)).
MIL is the smallest change that makes the question askable: a **bag of cells → one line label**
constrains only the aggregate, leaving the model free to differ between cells of the same line.

**The representation is part of the question, not a setting (Selin, 13.08.2026).** Q2 is asked of
`pca` and of `scgpt` separately and both arms run. One representation carrying within-line structure
while the other does not is a **result**, not a nuisance — it locates the structure in the encoding
rather than in the model.

### What Q2 claims here, and what it does not

Three questions had been running together under one name. Separated 13.08.2026:

| | | reachable with SCP542 + CTRPv2 |
|---|---|---|
| **a** | do per-cell predictions vary within a line, reproducibly, and not as a sequencing artifact? | **yes** — stages 0, 1, 2 and 6 below |
| **b** | is that variation *real* cellular heterogeneity of drug response? | **no** |
| **c** | does it predict *which* cells survive treatment? | **no** |

**(b) and (c) are out of reach for want of measurements, not because of this design.** No per-cell
response was ever measured — every label is one number per (cell line, drug) — and none of the four
primary sources carries post-treatment single-cell data. No threshold and no architecture closes that
gap; it takes a different dataset.

**So this notebook answers (a), and only (a).** That is worth having on its own terms: (a) is a
**necessary condition** for (b), so a model failing (a) has definitively not learned heterogeneity.
§3's program analysis is a **hint toward** (b) and explicitly not evidence for it. The write-up states
(b) and (c) as limitations of the data rather than leaving the narrowing implicit.

⚠️ **Candidate routes to (b) are recorded in
[Step 01](../docs/steps/01-datasets-and-harmonization.md#post-treatment-single-cell-data--what-would-be-needed-for-q2-b-and-what-exists)**
— MIX-seq, the palbociclib file already on disk, and lineage-barcoding studies, each with what it can
and cannot establish. **None is scoped, and none belongs to this notebook**; they are written down so
they are not rediscovered here as shortcuts.

## Decisions already taken

**Instance-level, not attention pooling (Selin, 12.08.2026).** Every cell gets its own *predicted
response*, and the line prediction aggregates them — rather than every cell getting an attention
*weight* over a pooled embedding. The two are the standard MIL alternatives (Ilse, Tomczak & Welling,
*Attention-based Deep Multiple Instance Learning*, ICML 2018): embedding-level usually predicts better,
instance-level is readable at the level of the individual instance. Q2 is a question about readability,
so the trade is taken deliberately and the cost in predictive performance is expected.

⚠️ **One consequence, recorded so it is not rediscovered:** selecting "the top-k cells" *by their
predicted value* and scoring that subset against the line's true response is biased by construction —
the extremes are shifted away from the line mean because they were chosen for being extreme. The
subpopulation-predictivity test that would have used it is therefore **not** in the criterion below.
It becomes available only if an attention weight is added alongside the per-cell predictions.

**Mean pooling, and the aggregator is never revisited (Selin, 13.08.2026).** The bag prediction is the
mean of its cells' predicted responses. It matches the synthetic control's own construction — a
mixture-weighted *average* label — and it is fixed now, before the model exists.

⚠️ **Its cost, taken deliberately.** Mean pooling under an MSE-family loss carries a real shrinkage
incentive: collapsing every cell onto the bag mean is a minimiser whenever the model cannot do better,
and nothing in the objective rewards spreading cells apart. Both alternatives were considered and
rejected — a **variance term in the loss** would make stage 1 pass by construction and would change the
loss, which [the governing rule](../docs/TODO.md) forbids; a **max or top-quantile aggregator** does not
force variation either (the max of equal per-cell values is that value) and mismatches an averaged
label.

**The retry is ruled out, and that is the part that binds.** If stage 7 fails, the aggregator is *not*
swapped for one that passes. *Fail → change the instrument → retry* is a forking path moved down one
level, and it would leave any subsequent positive unattributable. A stage-7 failure **ends the run**,
and what it means is fixed here rather than after the numbers are seen: **Q2 unanswered, the instrument
was not demonstrated** — never "no heterogeneity found".

**Same scorer as the per-cell model.** This notebook writes out-of-fold predictions in the shared
format — one row per cell line × drug × arm — so [`5_evaluation`](5_evaluation.ipynb) computes order,
top-of-order, values and spread for MIL and the per-cell model through identical code. The two are
comparable because they went through the same scorer, not because two notebooks agree by convention.

**Loss:** whatever `4a_percell_training` settles on, unchanged, so the architecture is the only thing
that moves ([the governing rule](../docs/TODO.md)). Ranking losses (RankNet, LambdaRank) become
well-posed *here* and nowhere earlier — they need one score per cell line, which is what a bag produces
— but they are a second change and belong to a later run, not this one.


## 2 · What counts as a positive Q2 result — fixed before the run

**There is no ground truth for within-line heterogeneity of drug response.** Every label is one number
per (cell line, drug); no per-cell response was ever measured, and SCP542 carries no post-treatment
single-cell data, so the ideal test — do the model's resistant cells match the cells that actually
survive treatment — cannot be run here. That is question **(b)** in §1, out of reach for want of
measurements.

What can be established is narrower and still worth having: **that the model's per-cell predictions
vary within a line, that the variation is reproducible rather than noise, and that it is not a
sequencing artifact.**

Five stages, in this order. Two are preconditions, one is a necessary condition, one is the test, one
is a veto.

| # | Stage | Role | Passes when |
|---|---|---|---|
| **0** | **Input ceiling** — within-line dispersion of the cell representations, measured on `pca` and `scgpt` separately, **before any training** | precondition on the **input** | dispersion clears the floor in §2.1 |
| **7** | **Synthetic positive control** — bags mixed from two cell lines of known, different response, labelled with the mixture-weighted value; scored **inside the bag**, source-A cells against source-B cells | precondition on the **instrument** | cell-level recovery clears `Q2_CONTROL_THRESHOLD` (§2.2) |
| **1** | **Spread** — within-line standard deviation of per-cell predicted responses | necessary condition | spread is a stated fraction of what stage 7 produced |
| **2** | **Reproducibility** — do independent seeds assign high and low predictions to the *same* cells? | **the test** | per-cell agreement across seeds exceeds the shuffled-cell control, by a stated fraction of stage 7's agreement |
| **6** | **Confound regression** — per-cell predictions against total counts, genes detected, mitochondrial fraction and cell-cycle score | **veto** | the confounds do *not* explain the variation |

**Why stage 0 comes first.** It costs no training. If the cells of a line collapse to a point in the
representation, no model of any kind can assign them different values, and every later stage is
measuring the wrong thing. It also separates two findings that a training run alone conflates: *the
input carries no within-line structure* and *the model did not use the structure that was there* — the
first is a statement about the representation, the second about the model. Because Q2 is asked of `pca`
and `scgpt` separately, stage 0 is per-representation and **may pass for one and fail for the other**,
which is itself a result.

**Why stage 7 comes before the rest.** Without it a negative is uninterpretable — "no heterogeneity
found" cannot be distinguished from "this method cannot find heterogeneity". With it, a negative
becomes a result: no detectable heterogeneity, by an instrument demonstrated to detect it when present.

**Why stage 7 is scored inside the bag and not on the bag (Selin, 13.08.2026).** The bag prediction is
the mean of its cells' predictions, so a model that assigns **every cell in the bag an identical value**
can land the bag mean exactly right and pass a bag-level test — while doing none of what stages 1, 2
and 6 go on to measure. Bag-level recovery would certify a shrunk instrument. It is still computed
(predicted bag value against mixture weight, across weights 0 … 1) and **reported as a diagnostic; it
cannot pass the stage.**

**Why stage 6 is a veto and not an analysis.** It looks descriptive, but it can turn a pass into a
fail: predictions that replicate across seeds *and* are explained by library size are a sequencing
artifact, not biology. Pre-registered here so it cannot become something run only when the answer is
unwelcome.

**Why only stages 0 and 7 need numbers chosen by judgement.** Stage 7 has ground truth, so it shows
what spread and what cross-seed agreement look like when heterogeneity is *definitely* present. Stages
1 and 2 are then expressed as fractions of what stage 7 actually produced — of stage 1's own quantity
measured on the control bags, and of stage 2's own — instead of thresholds invented in advance.

### 2.1 · ⬜ BLANK — how stage 0 measures within-line dispersion

> **Selin's.** What is measured, against what denominator, and where the floor sits.
>
> ```
> Q2_INPUT_DISPERSION = ...   # the statistic
> Q2_INPUT_FLOOR      = ...   # the value it must clear
> ```
>
> **A bare within-line dispersion is not interpretable** — its size depends on the representation's
> arbitrary scale, and `pca` and `scgpt` do not share one, so the two arms could not be compared. The
> natural denominator is **between-line** dispersion, which makes it a ratio and puts both arms on one
> scale; other denominators are possible and the choice belongs here. The statistic itself is open too
> — per-component standard deviation, mean distance to the line centroid, something else.
>
> ⚠️ **Timing.** The embeddings on disk predate the preprocessing corrections and R1 re-embeds, so a
> number computed before R1 is indicative, not final. The *shape* of the answer — spread versus
> collapsed — is unlikely to flip, but the floor should be set against the R1 embeddings.

### 2.2 · ⬜ BLANK — stage 7's quantity, its threshold, and the two fractions

> **Selin's, and the last thing needed before this notebook may be written.**
>
> ```
> Q2_CONTROL_QUANTITY  = ...   # A or B below
> Q2_CONTROL_THRESHOLD = ...   # the bar it must clear
> Q2_STAGE1_FRACTION   = ...   # f1, of stage 7's within-line spread
> Q2_STAGE2_FRACTION   = ...   # f2, of stage 7's cross-seed agreement
> ```
>
> **Two candidate quantities, both scored inside one mixed bag.** With two source lines per bag a rank
> correlation against the binary origin is a monotone transform of AUROC, so Spearman is not a third
> option — it *is* B.
>
> **A · recovered gap fraction** — `(mean prediction on A-cells − mean prediction on B-cells) / (y_A − y_B)`.
> On the label's own scale and directly interpretable ("recovers 60% of a known gap"). It is also the
> only one the project has an empirical anchor for: CTRPv2's repeated measurements disagree by a median
> of **0.49× the drug's spread across cell lines**
> ([Step 01](../docs/steps/01-datasets-and-harmonization.md), `outputs/archive/replicate_variation.csv`).
> That is *label* noise, not per-cell noise, so it does not set the bar — but it bounds it: if the
> labels themselves disagree by half a spread, requiring the model to recover more than ~0.5 of a known
> gap demands more precision than the labels contain. **Cost:** A is calibration-sensitive, and mean
> pooling gives the model a standing shrinkage incentive, so a model that orders cells correctly but
> pulls them toward the bag mean scores low — meaning A can fail stage 7 for the same reason stage 1
> would fail, which costs stage 7 its independence from the stage it is supposed to license.
>
> **B · per-cell AUROC** — are A-cells ranked above B-cells? Immune to shrinkage, so it isolates
> ordering, which is exactly what stages 1 and 2 are built on; stage 7 stays independent of stage 1.
> **Cost:** no scale. AUROC 0.6 says nothing about magnitude, and the 0.49 anchor does not translate
> into it.
>
> **On the value, whichever quantity is chosen.** The *form* of the bar can be principled — exceed a
> within-bag permutation null, shuffling which cells came from which source line — but a permutation
> bar is a significance bar, and with hundreds of cells per bag a statistically clear AUROC of 0.53 is
> still useless. The *magnitude* has no source in the literature that I have found. **An arbitrary
> magnitude documented as arbitrary is honest; the same number stated without comment reads as
> principled.** Whatever is chosen goes here with its reasoning, and with the two fractions.


## 3 · Closing analysis — what kind of cells were they?

Short and descriptive, and **it gates nothing**. By the time it runs, §2 has already decided whether Q2
is positive. This section says *what was found*, not *whether* something was found — the distinction
matters, because an enrichment discovered here cannot be promoted into evidence afterwards.

Two figures and a table:

**a · What the predictions track (stage 6, reported rather than vetoing).** The same regression the veto
uses, shown rather than thresholded: how much of the within-line variation in per-cell predictions is
explained by each of total counts, genes detected, mitochondrial fraction and cell-cycle score. If the
veto passed, these are all small, and showing them is what makes that credible.

**b · Which annotated programs the predictions track (stage 3).** Kinker et al. 2020 annotated recurrent
heterogeneity programs for this exact dataset, independently of any drug-response label. Both sides are
continuous — each cell has a program score, and instance-level MIL gives each cell a predicted response —
so **correlate the two across a line's cells**, per program, against a within-line permutation null.

*No top-k, decided 12.08.2026 (Selin).* An earlier draft took the most- and least-resistant predicted
cells and tested them for enrichment. Correlating the full continuous signal is strictly better here: it
needs no `k` to justify, uses every cell instead of a slice, and is the same method as (a) above — so the
confound check and the biology check are read on one scale rather than two.

⚠️ **Read the enrichment carefully.** Cell-cycle enrichment is close to guaranteed and would be weak
evidence of anything: the project already refuted *the cell-line effect is largely proliferation*
([Corrections](../docs/steps/corrections-and-dead-ends.md#the-cell-line-effect-is-largely-proliferation)),
and Kinker's two named associations are recorded as **not transferring to this task**
([Dead ends](../docs/steps/corrections-and-dead-ends.md#kinkers-two-named-associations-do-not-transfer-to-this-task)).
A program *other* than cell cycle would be the interesting outcome.

**c · One table** — per drug: does the cell ordering repeat across drugs, or is it drug-specific? A
general axis and a drug-specific subpopulation are different findings, and the table is the cheapest
way to tell them apart.

---

### What must not happen before the blanks in §2 are filled

No cells are added below this one. Building the model first and choosing the criterion afterwards is
the failure mode this stub exists to prevent, and it is the reason the criterion is written here in
prose rather than left to the run that will be scored by it.